In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
all_data=pd.read_csv("stock_data.csv")

In [4]:
all_data.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Price_change,20MA,50MA,Daily_Return,Volatility
0,2020-01-02,151.829529,160.619995,160.729996,158.330002,158.779999,22622100,MSFT,1.839996,NaN,NaN,NaN,NaN
1,2020-01-03,149.939026,158.619995,159.949997,158.059998,158.320007,21116200,MSFT,0.299988,NaN,NaN,-1.245175,NaN
2,2020-01-06,150.326569,159.029999,159.100006,156.509995,157.080002,20813700,MSFT,1.949997,NaN,NaN,0.258482,NaN
3,2020-01-07,148.955917,157.580002,159.669998,157.320007,159.320007,21634100,MSFT,-1.740005,NaN,NaN,-0.911776,NaN
4,2020-01-08,151.328568,160.089996,160.800003,157.949997,158.929993,27746500,MSFT,1.160004,NaN,NaN,1.592838,NaN


In [5]:
all_data["Date"]=pd.to_datetime(all_data["Date"])

In [6]:
all_data["Year"] = all_data["Date"].dt.year

In [7]:
#KPI - 1 ANNUAL RETURN

annual_return = (all_data.groupby(["Ticker","Year"]).agg( Start_Price=("Close","first"), End_Price=("Close","last") ))

In [8]:
annual_return["Annual_Return"] = (
    (
        annual_return["End_Price"]
        -
        annual_return["Start_Price"]
    )
    /
    annual_return["Start_Price"]
) * 100

In [9]:
annual_return["Annual_Return"]=annual_return["Annual_Return"].round(2)

In [10]:
annual_return.head()

Start_Price   End_Price  Annual_Return
Ticker Year                                        
AAPL   2020    75.087502  132.690002          76.71
       2021   129.410004  177.570007          37.22
       2022   182.009995  129.929993         -28.61
       2023   125.070000  192.529999          53.94
       2024   185.639999  250.419998          34.90

In [11]:
all_data.tail()

,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Price_change,20MA,50MA,Daily_Return,Volatility,Year
7530,2025-12-23,313.941345,314.350006,314.940002,309.320007,309.630005,25478700,GOOGL,4.720001,313.539000,6.951175,1.475243,1.517901,2025
7531,2025-12-24,313.681671,314.089996,315.079987,311.920013,314.769989,10097400,GOOGL,-0.679993,313.071500,6.553266,-0.082713,1.471613,2025
7532,2025-12-26,313.102478,313.510010,315.089996,312.279999,314.480011,10899000,GOOGL,-0.970001,312.749500,6.352642,-0.184656,1.454950,2025
7533,2025-12-29,313.152405,313.559998,314.019989,310.619995,311.369995,19621800,GOOGL,2.190002,312.418500,6.113052,0.015945,1.454673,2025
7534,2025-12-30,313.442017,313.850006,316.950012,312.459991,312.500000,17380900,GOOGL,1.350006,312.366499,6.095319,0.092489,1.407892,2025


In [51]:
cagr = (
    all_data.groupby("Ticker")
    .agg(
        Start_Price=("Close", "first"),
        End_Price=("Close", "last"),
        First_Date=("Date", "first"),
        Last_Date=("Date", "last")
    )
    .reset_index()
)

cagr["Years"] = (cagr["Last_Date"] - cagr["First_Date"]).dt.days / 365.25

cagr["CAGR"] = (
    (cagr["End_Price"] / cagr["Start_Price"]) ** (1 / cagr["Years"]) - 1
) * 100

cagr["CAGR"] = cagr["CAGR"].round(2)

cagr = cagr[["Ticker", "CAGR"]]

In [52]:
cagr.head()

,Ticker,CAGR
0,AAPL,24.04
1,AMZN,16.13
2,GOOGL,28.93
3,MSFT,20.35
4,NVDA,77.61


In [15]:
annual_return.head()

Start_Price   End_Price  Annual_Return
Ticker Year                                        
AAPL   2020    75.087502  132.690002          76.71
       2021   129.410004  177.570007          37.22
       2022   182.009995  129.929993         -28.61
       2023   125.070000  192.529999          53.94
       2024   185.639999  250.419998          34.90

In [16]:
#KPI 3 MAXIMUM DRAWDOWN

all_data=all_data.sort_values(["Ticker","Date"]) #sorting values first
all_data["Running_max"]=all_data.groupby("Ticker")["Close"].cummax() #finding the max value
all_data["Drawdown"]=((all_data["Close"]-all_data["Running_max"])/all_data["Running_max"])*100 #max drawdown formulae

In [17]:
max_drawdown=(all_data.groupby("Ticker")["Drawdown"].min().reset_index().rename(columns={"Drawdown":"Max_Drawdown"}))
max_drawdown["Max_Drawdown"]=max_drawdown["Max_Drawdown"].round(2)

In [18]:
max_drawdown.head()

,Ticker,Max_Drawdown
0,AAPL,-33.43
1,AMZN,-56.15
2,GOOGL,-44.32
3,MSFT,-37.56
4,NVDA,-66.36


In [19]:
all_data.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Price_change,20MA,50MA,Daily_Return,Volatility,Year,Running_max,Drawdown
1507,2020-01-02,72.333855,75.087502,75.150002,73.797501,74.059998,135480400,AAPL,1.027504,NaN,NaN,NaN,NaN,2020,75.087502,0.000000
1508,2020-01-03,71.630646,74.357498,75.144997,74.125000,74.287498,146322800,AAPL,0.070000,NaN,NaN,-0.972204,NaN,2020,75.087502,-0.972204
1509,2020-01-06,72.201416,74.949997,74.989998,73.187500,73.447502,118387200,AAPL,1.502495,NaN,NaN,0.796825,NaN,2020,75.087502,-0.183126
1510,2020-01-07,71.861847,74.597504,75.224998,74.370003,74.959999,108872000,AAPL,-0.362495,NaN,NaN,-0.470305,NaN,2020,75.087502,-0.652569
1511,2020-01-08,73.017853,75.797501,76.110001,74.290001,74.290001,132079200,AAPL,1.507500,NaN,NaN,1.608629,NaN,2020,75.797501,0.000000


In [20]:
cagr.head()

,Ticker,CAGR
0,AAPL,24.04
1,AMZN,16.13
2,GOOGL,28.93
3,MSFT,20.35
4,NVDA,77.61


In [21]:
annual_return.head()

Start_Price   End_Price  Annual_Return
Ticker Year                                        
AAPL   2020    75.087502  132.690002          76.71
       2021   129.410004  177.570007          37.22
       2022   182.009995  129.929993         -28.61
       2023   125.070000  192.529999          53.94
       2024   185.639999  250.419998          34.90

In [22]:
#KPI 4 VOLATILITY

volatility= (all_data.groupby("Ticker")["Daily_Return"].std().reset_index().rename(columns={"Daily_Return":"volatility"}))

In [23]:
volatility["volatility"]=volatility["volatility"].round(2)

In [24]:
volatility.head()

,Ticker,volatility
0,AAPL,2.00
1,AMZN,2.25
2,GOOGL,2.05
3,MSFT,1.86
4,NVDA,3.35


In [25]:
#kpi 5 liquidity
liquidity = (
    all_data.groupby("Ticker")["Volume"]
    .mean()
    .reset_index()
    .rename(columns={"Volume": "Avg_Liquidity"})
)

liquidity["Avg_Liquidity"] = liquidity["Avg_Liquidity"].round(0)

liquidity

,Ticker,Avg_Liquidity
0,AAPL,84575324.0
1,AMZN,64528216.0
2,GOOGL,33625452.0
3,MSFT,27577351.0
4,NVDA,409494245.0


In [26]:
#kpi 6 sharpe ratio
avg_return = (
    all_data.groupby("Ticker")["Daily_Return"]
    .mean()
    .reset_index()
    .rename(columns={"Daily_Return": "Avg_Daily_Return"})
)

avg_return["Avg_Daily_Return"] = avg_return["Avg_Daily_Return"].round(4)

avg_return

,Ticker,Avg_Daily_Return
0,AAPL,0.1058
1,AMZN,0.0848
2,GOOGL,0.1221
3,MSFT,0.0911
4,NVDA,0.2846


In [27]:
sharpe = avg_return.merge(volatility, on="Ticker")

In [28]:
sharpe["Sharpe_Ratio"] = sharpe["Avg_Daily_Return"] / sharpe["volatility"]

In [29]:
sharpe["Sharpe_Ratio"] = sharpe["Sharpe_Ratio"].round(4)
sharpe

,Ticker,Avg_Daily_Return,volatility,Sharpe_Ratio
0,AAPL,0.1058,2.00,0.0529
1,AMZN,0.0848,2.25,0.0377
2,GOOGL,0.1221,2.05,0.0596
3,MSFT,0.0911,1.86,0.0490
4,NVDA,0.2846,3.35,0.0850


In [30]:
risk = volatility.merge(max_drawdown, on="Ticker")
risk["Drawdown_Abs"] = risk["Max_Drawdown"].abs()

In [31]:
risk["Volatility_Norm"] = (
    (risk["volatility"] - risk["volatility"].min()) /
    (risk["volatility"].max() - risk["volatility"].min())
)

risk["Drawdown_Norm"] = (
    (risk["Drawdown_Abs"] - risk["Drawdown_Abs"].min()) /
    (risk["Drawdown_Abs"].max() - risk["Drawdown_Abs"].min())
)

In [32]:
risk["Risk_Score"] = (
    0.6 * risk["Volatility_Norm"] +
    0.4 * risk["Drawdown_Norm"]
) * 100

risk["Risk_Score"] = risk["Risk_Score"].round(2)

In [33]:
cagr=cagr.reset_index()

In [34]:
cagr=cagr=[["Ticker","cagr"]]

In [35]:
max_drawdown = max_drawdown[["Ticker", "Max_Drawdown"]]

In [36]:
volatility = volatility[["Ticker", "volatility"]]

In [37]:
liquidity = liquidity[["Ticker", "Avg_Liquidity"]]

In [38]:
sharpe = sharpe[["Ticker", "Sharpe_Ratio"]]

In [39]:
risk = risk[["Ticker", "Risk_Score"]]

In [40]:
avg_annual_return = (
    annual_return.groupby("Ticker")["Annual_Return"]
    .mean()
    .reset_index()
    .rename(columns={"Annual_Return": "Avg_Annual_Return"})
)

avg_annual_return["Avg_Annual_Return"] = avg_annual_return["Avg_Annual_Return"].round(2)

In [53]:
print(type(cagr))

<class 'pandas.core.frame.DataFrame'>


In [50]:
print(type(max_drawdown))

<class 'pandas.core.frame.DataFrame'>


In [43]:
print(type(volatility))

<class 'pandas.core.frame.DataFrame'>


In [44]:
print(type(liquidity))

<class 'pandas.core.frame.DataFrame'>


In [45]:
print(type(sharpe))

<class 'pandas.core.frame.DataFrame'>


In [46]:
print(type(risk))

<class 'pandas.core.frame.DataFrame'>


In [54]:
final_kpi = (
    cagr
    .merge(max_drawdown, on="Ticker", how="left")
    .merge(volatility, on="Ticker", how="left")
    .merge(liquidity, on="Ticker", how="left")
    .merge(sharpe, on="Ticker", how="left")
    .merge(risk, on="Ticker", how="left")
    .merge(avg_annual_return, on="Ticker", how="left")
)

In [55]:
final_kpi = final_kpi[
    [
        "Ticker",
        "CAGR",
        "Avg_Annual_Return",
        "Max_Drawdown",
        "volatility",
        "Sharpe_Ratio",
        "Avg_Liquidity",
        "Risk_Score"
    ]
]

In [56]:
final_kpi.head()

,Ticker,CAGR,Avg_Annual_Return,Max_Drawdown,volatility,Sharpe_Ratio,Avg_Liquidity,Risk_Score
0,AAPL,24.04,31.02,-33.43,2.00,0.0529,84575324.0,5.64
1,AMZN,16.13,25.75,-56.15,2.25,0.0377,64528216.0,43.30
2,GOOGL,28.93,36.03,-44.32,2.05,0.0596,33625452.0,20.88
3,MSFT,20.35,25.28,-37.56,1.86,0.0490,27577351.0,5.02
4,NVDA,77.61,108.46,-66.36,3.35,0.0850,409494245.0,100.00


In [57]:
# Best CAGR
final_kpi.sort_values("CAGR", ascending=False)

# Lowest risk
final_kpi.sort_values("Risk_Score")

# Best Sharpe ratio
final_kpi.sort_values("Sharpe_Ratio", ascending=False)

# Worst drawdown
final_kpi.sort_values("Max_Drawdown")

,Ticker,CAGR,Avg_Annual_Return,Max_Drawdown,volatility,Sharpe_Ratio,Avg_Liquidity,Risk_Score
4,NVDA,77.61,108.46,-66.36,3.35,0.0850,409494245.0,100.00
1,AMZN,16.13,25.75,-56.15,2.25,0.0377,64528216.0,43.30
2,GOOGL,28.93,36.03,-44.32,2.05,0.0596,33625452.0,20.88
3,MSFT,20.35,25.28,-37.56,1.86,0.0490,27577351.0,5.02
0,AAPL,24.04,31.02,-33.43,2.00,0.0529,84575324.0,5.64


In [58]:
investor_view = pd.DataFrame({
    "Investor_Type": ["Conservative", "Balanced", "Aggressive"],
    "Suggested_Stock": ["MSFT", "AAPL / GOOGL", "NVDA"],
    "Reason": [
        "Lower volatility, lower drawdown, and relatively low risk score.",
        "Good mix of return and manageable risk.",
        "Highest CAGR and return potential, but comes with the highest volatility and drawdown risk."
    ]
})

investor_view

,Investor_Type,Suggested_Stock,Reason
0,Conservative,MSFT,"Lower volatility, lower drawdown, and relative..."
1,Balanced,AAPL / GOOGL,Good mix of return and manageable risk.
2,Aggressive,NVDA,"Highest CAGR and return potential, but comes w..."


In [59]:
final_kpi.to_csv("final_kpi.csv", index=False)

In [61]:
stock_timeseries = all_data[
    ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume",
     "Daily_Return", "20MA", "50MA", "Volatility", "Year"]
]

stock_timeseries.to_csv("stock_timeseries.csv", index=False)